# AWFedAvg — Experimental Results Figures
All figures from the paper, auto-generated from experiment JSON files.
Run `python experiments.py` first to produce the JSON result files.

In [ ]:
import json, os, math, warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')

# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.labelsize':   12,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
    'legend.fontsize':  10,
    'figure.dpi':       150,
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

RESULTS_DIR = 'results'
FIGS_DIR    = 'figures'
os.makedirs(FIGS_DIR, exist_ok=True)

# Colour palette (colour-blind friendly)
PALETTE = ['#1F77B4','#FF7F0E','#2CA02C','#D62728','#9467BD',
           '#8C564B','#E377C2','#7F7F7F','#BCBD22','#17BECF']

def load(fname):
    path = os.path.join(RESULTS_DIR, fname)
    if not os.path.exists(path):
        # Try current directory
        path = fname
    if not os.path.exists(path):
        print(f'⚠  {fname} not found — skipping plot')
        return []
    with open(path) as f:
        return json.load(f)

def savefig(name):
    path = os.path.join(FIGS_DIR, name)
    plt.savefig(path, bbox_inches='tight', dpi=150)
    print(f'  ✅ Saved → {path}')
    plt.show()

print('Setup complete.')

## Figure 1 — Ablation Study: Reward per Component

In [ ]:
data = load('ablation_results.json')
if data:
    names   = [r['name'] for r in data]
    means   = [r['reward_mean']  for r in data]
    stds    = [r['reward_std']   for r in data]
    ci95    = [r['reward_ci95']  for r in data]
    cohens  = [r['cohens_d']     for r in data]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # --- Bar chart with CI error bars ---
    ax = axes[0]
    colors = [PALETTE[4] if 'Full' in n else PALETTE[0] for n in names]
    bars = ax.bar(range(len(names)), means, yerr=ci95,
                  color=colors, alpha=0.85, capsize=5, width=0.6,
                  error_kw={'elinewidth': 1.5})
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels([n.replace(' (', '\n(') for n in names], rotation=15, ha='right')
    ax.set_ylabel('Final Global Reward (mean ± 95% CI)')
    ax.set_title('Ablation Study — Contribution of Each Component')
    for i, (m, ci) in enumerate(zip(means, ci95)):
        ax.text(i, m + ci + 0.005, f'{m:.4f}', ha='center', fontsize=9, fontweight='bold')

    # --- Cohen's d bar chart ---
    ax2 = axes[1]
    cd_vals  = [0.0] + cohens[1:]   # baseline = 0 by definition
    cd_colors = ['#2CA02C' if v >= 0 else '#D62728' for v in cd_vals]
    ax2.bar(range(len(names)), cd_vals, color=cd_colors, alpha=0.85, width=0.6)
    ax2.axhline(0, color='black', linewidth=0.8)
    ax2.axhline( 0.2, linestyle='--', color='gray', alpha=0.5, label='small effect')
    ax2.axhline( 0.5, linestyle='--', color='orange', alpha=0.5, label='medium effect')
    ax2.axhline( 0.8, linestyle='--', color='red', alpha=0.5, label='large effect')
    ax2.set_xticks(range(len(names)))
    ax2.set_xticklabels([n.replace(' (', '\n(') for n in names], rotation=15, ha='right')
    ax2.set_ylabel("Cohen's d vs FL Only baseline")
    ax2.set_title("Effect Size vs Baseline")
    ax2.legend(loc='lower right', fontsize=8)

    plt.tight_layout()
    savefig('fig1_ablation.pdf')

## Figure 2 — Scalability: Overhead vs K

In [ ]:
data = load('scalability_results.json')
if data:
    K_vals   = [r['num_clients']   for r in data]
    rewards  = [r['reward_mean']   for r in data]
    rwd_std  = [r['reward_std']    for r in data]
    bc_times = [r['bc_mean']       for r in data]
    ipfs_kb  = [r['ipfs_mean']     for r in data]
    wall     = [r['wall_mean']     for r in data]
    eps_tot  = [r['eps_total']     for r in data]

    fig, axes = plt.subplots(2, 2, figsize=(13, 10))

    # 2a — Reward vs K
    ax = axes[0,0]
    ax.errorbar(K_vals, rewards, yerr=rwd_std, marker='o', color=PALETTE[0],
                linewidth=2, capsize=4, markersize=7, label='Reward')
    ax.set_xlabel('Number of clients (K)')
    ax.set_ylabel('Final Global Reward')
    ax.set_title('(a) Convergence Quality vs K')
    ax.set_xticks(K_vals)

    # 2b — BC latency vs K
    ax = axes[0,1]
    ax.bar(range(len(K_vals)), bc_times, color=PALETTE[1], alpha=0.8, width=0.5)
    ax.set_xticks(range(len(K_vals)))
    ax.set_xticklabels([f'K={k}' for k in K_vals])
    ax.set_ylabel('Blockchain Latency (s/round)')
    ax.set_title('(b) Blockchain Overhead vs K  [O(K)]')

    # 2c — IPFS upload vs K
    ax = axes[1,0]
    ax.bar(range(len(K_vals)), ipfs_kb, color=PALETTE[2], alpha=0.8, width=0.5)
    ax.set_xticks(range(len(K_vals)))
    ax.set_xticklabels([f'K={k}' for k in K_vals])
    ax.set_ylabel('IPFS Upload (KB/round)')
    ax.set_title('(c) IPFS Bandwidth vs K  [O(|w|), independent of K]')

    # 2d — Wall clock vs K
    ax = axes[1,1]
    ax.plot(K_vals, wall, marker='s', color=PALETTE[3], linewidth=2, markersize=7)
    ax.fill_between(K_vals,
                    [w - r['wall_std'] for w, r in zip(wall, data)],
                    [w + r['wall_std'] for w, r in zip(wall, data)],
                    alpha=0.15, color=PALETTE[3])
    ax.set_xlabel('Number of clients (K)')
    ax.set_ylabel('Wall-clock Time (s/round)')
    ax.set_title('(d) Training Wall Time vs K')
    ax.set_xticks(K_vals)

    plt.suptitle('Scalability Analysis — Full System (K ∈ {3,5,10,20,50})', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    savefig('fig2_scalability.pdf')

## Figure 3 — Gas Cost vs K (from estimate_gas.py output)

In [ ]:
gas_data = load('gas_report.json')
if gas_data and 'scalability' in gas_data:
    sc = gas_data['scalability']
    Ks   = [r['K']        for r in sc]
    gas  = [r['gas_round'] for r in sc]
    usd  = [r['usd_round'] for r in sc]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax = axes[0]
    ax.bar(range(len(Ks)), gas, color=PALETTE[5], alpha=0.85, width=0.5)
    ax.set_xticks(range(len(Ks)))
    ax.set_xticklabels([f'K={k}' for k in Ks])
    ax.set_ylabel('Gas Units per Round')
    ax.set_title('(a) Gas per Round vs K\n(mandatory txns only)')
    for i, g in enumerate(gas):
        ax.text(i, g + 500, f'{g:,}', ha='center', fontsize=9)

    ax = axes[1]
    ax.bar(range(len(Ks)), [u * 1e6 for u in usd], color=PALETTE[6], alpha=0.85, width=0.5)
    ax.set_xticks(range(len(Ks)))
    ax.set_xticklabels([f'K={k}' for k in Ks])
    ax.set_ylabel('Cost per Round (µUSD)')
    ax.set_title(f'(b) USD Cost per Round vs K\n({gas_data["config"]["gas_price_gwei"]} Gwei, ETH=${gas_data["config"]["eth_usd"]})')

    plt.suptitle('Gas Cost Analysis — O(K) Blockchain Complexity', fontsize=13, fontweight='bold')
    plt.tight_layout()
    savefig('fig3_gas_cost.pdf')

## Figure 4 — Privacy Trade-off: Reward vs ε

In [ ]:
data = load('privacy_results.json')
if data:
    eps_vals = [float(r['name'].split('ε=')[-1]) for r in data]
    rewards  = [r['reward_mean'] for r in data]
    stds     = [r['reward_std']  for r in data]
    ci95     = [r['reward_ci95'] for r in data]
    eps_tot  = [r['eps_total']   for r in data]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax = axes[0]
    ax.errorbar(eps_vals, rewards, yerr=ci95, marker='o', color=PALETTE[0],
                linewidth=2, capsize=5, markersize=8, label='Reward ± 95% CI')
    ax.fill_between(eps_vals,
                    [m-s for m,s in zip(rewards, stds)],
                    [m+s for m,s in zip(rewards, stds)],
                    alpha=0.12, color=PALETTE[0], label='± std')
    ax.set_xscale('log')
    ax.set_xlabel('Privacy Budget ε (log scale)  ←  stronger privacy')
    ax.set_ylabel('Final Global Reward')
    ax.set_title('(a) Reward vs Privacy Budget ε')
    ax.legend()
    ax.invert_xaxis()  # strong privacy (small ε) on the left

    ax2 = axes[1]
    ax2.plot(eps_vals, eps_tot, marker='s', color=PALETTE[1], linewidth=2, markersize=8)
    ax2.set_xscale('log')
    ax2.set_xlabel('ε per round (log scale)')
    ax2.set_ylabel('ε_total = √(2T·ln(1/δ))·ε')
    ax2.set_title('(b) Cumulative Privacy Budget ε_total\n(Advanced Composition Theorem)')
    ax2.invert_xaxis()

    plt.suptitle('Privacy–Utility Trade-off', fontsize=13, fontweight='bold')
    plt.tight_layout()
    savefig('fig4_privacy_tradeoff.pdf')

## Figure 5 — Attack Robustness

In [ ]:
data = load('attack_results.json')
if data:
    names    = [r['name']        for r in data]
    rewards  = [r['reward_mean'] for r in data]
    ci95     = [r['reward_ci95'] for r in data]
    cohens   = [r['cohens_d']    for r in data]
    baseline_r = rewards[0]

    # Categorise attacks
    attack_cats = {
        'No Attack': '#2CA02C',
        'Byzantine': '#D62728',
        'Poisoning': '#FF7F0E',
        'Free-rider': '#9467BD',
        'Collusion':  '#8C564B',
        'Replay':     '#17BECF',
        'Sybil':      '#E377C2',
    }
    def get_color(name):
        for k, c in attack_cats.items():
            if k.lower() in name.lower():
                return c
        return PALETTE[7]

    colors = [get_color(n) for n in names]
    delta_pct = [(r - baseline_r) / (abs(baseline_r) + 1e-9) * 100 for r in rewards]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # 5a — Absolute reward
    ax = axes[0]
    bars = ax.bar(range(len(names)), rewards, yerr=ci95, color=colors,
                  alpha=0.85, capsize=4, width=0.65,
                  error_kw={'elinewidth': 1.5})
    ax.axhline(baseline_r, linestyle='--', color='black', linewidth=1.2,
               label=f'No-attack baseline ({baseline_r:.4f})')
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels([n.replace(' ', '\n') for n in names],
                        rotation=15, ha='right', fontsize=9)
    ax.set_ylabel('Final Global Reward (mean ± 95% CI)')
    ax.set_title('(a) Reward Under Each Attack Scenario\n(Full System with all defences active)')
    ax.legend(fontsize=9)

    # 5b — Degradation %
    ax = axes[1]
    bar_colors = ['#2CA02C' if d >= 0 else '#D62728' for d in delta_pct]
    ax.bar(range(len(names)), delta_pct, color=bar_colors, alpha=0.85, width=0.65)
    ax.axhline(0, color='black', linewidth=0.9)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels([n.replace(' ', '\n') for n in names],
                        rotation=15, ha='right', fontsize=9)
    ax.set_ylabel('Reward Degradation vs No-Attack (%)')
    ax.set_title('(b) Relative Reward Degradation (%)')
    for i, d in enumerate(delta_pct):
        ax.text(i, d + (0.3 if d >= 0 else -1.2), f'{d:+.1f}%', ha='center', fontsize=8)

    # Legend
    handles = [mpatches.Patch(color=c, label=k) for k, c in attack_cats.items()]
    axes[0].legend(handles=handles, loc='lower right', fontsize=8, title='Attack type')

    plt.suptitle('Attack Robustness — AWFedAvg + DP + SecAgg + Reputation Defences',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    savefig('fig5_attacks.pdf')

## Figure 6 — Summary Tables (LaTeX-ready)

In [ ]:
def latex_table(data, title, cols, fmt):
    """Print a LaTeX table fragment."""
    if not data:
        return
    ncol = len(cols)
    col_spec = 'l' + 'r' * (ncol - 1)
    print(f'% {title}')
    print(f'\\begin{{tabular}}{{{col_spec}}}')
    print('\\toprule')
    print(' & '.join(cols) + ' \\\\')
    print('\\midrule')
    for r in data:
        vals = [fmt[k](r[k]) if k in r else '--' for k in fmt]
        print(' & '.join(str(v) for v in vals) + ' \\\\')
    print('\\bottomrule')
    print('\\end{tabular}')
    print()

# Table 3 — Ablation
abl = load('ablation_results.json')
if abl:
    print('='*60)
    print('TABLE 3 — Ablation Study (K=3, T=10, 10 seeds)')
    print('='*60)
    latex_table(
        abl,
        'Ablation Study',
        cols=['Configuration', 'Reward', '± Std', '95\\% CI', 'BC (s)', 'IPFS (KB)', r'$\varepsilon_{\rm total}$', "Cohen's $d$"],
        fmt={
            'name':        lambda v: v,
            'reward_mean': lambda v: f'{v:.4f}',
            'reward_std':  lambda v: f'{v:.4f}',
            'reward_ci95': lambda v: f'{v:.4f}',
            'bc_mean':     lambda v: f'{v:.3f}',
            'ipfs_mean':   lambda v: f'{v:.1f}',
            'eps_total':   lambda v: f'{v:.4f}',
            'cohens_d':    lambda v: f'{v:+.3f}',
        }
    )

# Table 4 — Scalability
sc = load('scalability_results.json')
if sc:
    print('='*60)
    print('TABLE 4 — Scalability (Full System, 10 seeds)')
    print('='*60)
    latex_table(
        sc,
        'Scalability',
        cols=['K', 'T', 'Reward', '± Std', 'BC (s)', 'IPFS (KB)', r'$\varepsilon_{\rm total}$', 'Wall (s)'],
        fmt={
            'num_clients': lambda v: str(v),
            'num_rounds':  lambda v: str(v),
            'reward_mean': lambda v: f'{v:.4f}',
            'reward_std':  lambda v: f'{v:.4f}',
            'bc_mean':     lambda v: f'{v:.3f}',
            'ipfs_mean':   lambda v: f'{v:.1f}',
            'eps_total':   lambda v: f'{v:.4f}',
            'wall_mean':   lambda v: f'{v:.1f}',
        }
    )

# Table 5 — Privacy trade-off
pv = load('privacy_results.json')
if pv:
    print('='*60)
    print('TABLE 5 — Privacy Trade-off (10 seeds)')
    print('='*60)
    latex_table(
        pv,
        'Privacy Trade-off',
        cols=[r'$\varepsilon$/round', 'Reward', '± Std', r'$\varepsilon_{\rm total}$', "Cohen's $d$"],
        fmt={
            'name':        lambda v: v.split('=')[-1],
            'reward_mean': lambda v: f'{v:.4f}',
            'reward_std':  lambda v: f'{v:.4f}',
            'eps_total':   lambda v: f'{v:.4f}',
            'cohens_d':    lambda v: f'{v:+.3f}',
        }
    )

# Table 6 — Attacks
atk = load('attack_results.json')
if atk:
    print('='*60)
    print('TABLE 6 — Attack Robustness (10 seeds)')
    print('='*60)
    base_r = atk[0]['reward_mean'] if atk else 1.0
    for r in atk:
        r['delta_pct'] = (r['reward_mean'] - base_r) / (abs(base_r) + 1e-9) * 100
    latex_table(
        atk,
        'Attack Robustness',
        cols=['Attack', 'Reward', '± Std', r'$\Delta$ Reward (\%)', "Cohen's $d$"],
        fmt={
            'name':        lambda v: v,
            'reward_mean': lambda v: f'{v:.4f}',
            'reward_std':  lambda v: f'{v:.4f}',
            'delta_pct':   lambda v: f'{v:+.2f}',
            'cohens_d':    lambda v: f'{v:+.3f}',
        }
    )

## Figure 7 — Convergence Curve (per-round rewards, if available)

In [ ]:
abl = load('ablation_results.json')
# Each result has runs[*].per_round[*].average_reward — aggregate across seeds
if abl:
    fig, ax = plt.subplots(figsize=(10, 5))
    for idx, cfg in enumerate(abl):
        # Collect per-round rewards across all successful runs
        all_runs_rewards = []
        for run in cfg.get('runs', []):
            if run.get('error'): continue
            rw = [p.get('average_reward', None) for p in run.get('per_round', [])]
            rw = [r for r in rw if r is not None]
            if rw:
                all_runs_rewards.append(rw)
        if not all_runs_rewards:
            continue
        # Pad to same length
        T = max(len(r) for r in all_runs_rewards)
        padded = np.array([r + [r[-1]] * (T - len(r)) for r in all_runs_rewards])
        mean = padded.mean(axis=0)
        std  = padded.std(axis=0)
        rounds = np.arange(1, T + 1)
        color = PALETTE[idx % len(PALETTE)]
        ax.plot(rounds, mean, color=color, linewidth=2, label=cfg['name'])
        ax.fill_between(rounds, mean - std, mean + std, alpha=0.12, color=color)

    ax.set_xlabel('Federated Round')
    ax.set_ylabel('Global Reward (mean ± std, 10 seeds)')
    ax.set_title('Convergence Curves — Ablation Study')
    ax.legend(loc='lower right', fontsize=9)
    plt.tight_layout()
    savefig('fig7_convergence.pdf')
    print('Note: populate per_round data from experiments for full convergence curves.')

## Done
All figures saved to `figures/` directory.

In [ ]:
figs = [f for f in os.listdir(FIGS_DIR) if f.endswith('.pdf')] if os.path.isdir(FIGS_DIR) else []
print(f'Generated {len(figs)} figure files in ./{FIGS_DIR}/')
for f in sorted(figs):
    print(f'  ✅ {f}')